In [1]:
import sqlite3
import pandas as pd
import uuid

In [2]:
data_excel_path = "D:/applications/Projets/Mondelez/BDD FINAL_ PLANTING 2025_vf_on farm.xlsx"
data_frame = pd.read_excel(data_excel_path, sheet_name=1)

In [3]:
# Connexion à une base (crée le fichier s'il n'existe pas)

conn = sqlite3.connect("db_mondelez_last.sqlite3")

# Crée un curseur pour exécuter des requêtes SQL
cursor = conn.cursor()

In [4]:
query_select_producteurs = "SELECT * FROM myapi_producteur WHERE campagne_id = 5 AND section_id IN (SELECT id FROM myapi_section WHERE cooperative_id = 41)"
query_select_parcelles = f"SELECT * FROM myapi_parcelle WHERE producteur_id IN ({query_select_producteurs.replace('SELECT *', 'SELECT code')})"
query_select_planting = f"SELECT * FROM myapi_planting WHERE parcelle_id IN ({query_select_parcelles.replace('SELECT *', 'SELECT code')})"
query_select_planting_created_today = f"SELECT * FROM myapi_planting WHERE created_at > '2025-12-02 00:00:00'"
query_select_detail_planting_created_today = f"SELECT * FROM myapi_detailplanting WHERE created_at > '2025-12-02 00:00:00'"

In [5]:
# Selection des données
cursor.execute(query_select_producteurs)
all_producteurs = cursor.fetchall()

cursor.execute(query_select_parcelles)
all_parcelles = cursor.fetchall()

cursor.execute(query_select_planting)
all_planting = cursor.fetchall()

cursor.execute(query_select_planting_created_today)
all_planting_created_today = cursor.fetchall()

cursor.execute(query_select_detail_planting_created_today)
all_detail_planting_created_today = cursor.fetchall()

print(f"Nombre de producteurs : {len(all_producteurs)}\nNombre de parcelles : {len(all_parcelles)}\n Nombre de planting : {len(all_planting)}\n Nombre de planting crées aujourd'hui : {len(all_planting_created_today)}\nNombre de plants: {sum([planting[2] for planting in all_planting_created_today])}\n Nombre de detail planting crées aujourd'hui : {len(all_detail_planting_created_today)}")

Nombre de producteurs : 1
Nombre de parcelles : 0
 Nombre de planting : 0
 Nombre de planting crées aujourd'hui : 0
Nombre de plants: 0
 Nombre de detail planting crées aujourd'hui : 0


In [6]:
print(all_producteurs)

[('CIV-GOH-A01200309-009-0001', 'GOUHOUN GOUAMENE EPHRAIM', 'None', '', 0, 1, 'None', None, 1, '2025-12-03 00:47:56.000708', '2025-12-03 00:47:56.010674', 5, None, 627, 'EB2BECECC4')]


In [8]:
# Suppression de données
nbre_planting_supprimes = cursor.execute(query_select_planting_created_today.replace("SELECT *", "DELETE"))
nbre_parcelles_supprimees = cursor.execute(query_select_parcelles.replace("SELECT *", "DELETE"))
nbre_producteurs_supprimes = cursor.execute(query_select_producteurs.replace("SELECT *", "DELETE"))
nbre_detail_planting_supprimes = cursor.execute(query_select_detail_planting_created_today.replace("SELECT *", "DELETE"))
conn.commit()
print(f"Nombre de planting supprimés : {nbre_planting_supprimes.fetchone()}\nNombre de parcelles supprimées : {nbre_parcelles_supprimees}\nNombre de producteurs supprimés : {nbre_producteurs_supprimes}\nNombre de detail planting supprimés : {nbre_detail_planting_supprimes}")

Nombre de planting supprimés : None
Nombre de parcelles supprimées : <sqlite3.Cursor object at 0x0000024F0B1F2F40>
Nombre de producteurs supprimés : <sqlite3.Cursor object at 0x0000024F0B1F2F40>
Nombre de detail planting supprimés : <sqlite3.Cursor object at 0x0000024F0B1F2F40>


In [35]:
# Nombre de producteurs dans le dataFrame
print(f"Nombre de producteurs dans le dataFrame : {data_frame['CODE PRODUCTEUR'].nunique()}")

Nombre de producteurs dans le dataFrame : 497


In [20]:
# Parcelles du dataFrame qui ne sont pas dans la base de données
# parcelles_non_enregistres = [code_parcelle for (code_parcelle, *_) in all_parcelles if code_parcelle not in data_frame['CODE PARCELLE'].values]
parcelles_non_enregistres = [code_parcelle for code_parcelle in data_frame['CODE PARCELLE'].values if code_parcelle not in [code for (code, *_) in all_parcelles]]
print(f"Parcelles non enregistrées : {parcelles_non_enregistres} \n'({len(parcelles_non_enregistres)})")

Parcelles non enregistrées : [nan] 
'(1)


AttributeError: attribute '__default__' of 'typing.ParamSpec' objects is not writable